# Analyzing Research Publication Trends with Polars

In this notebook we'll load a synthetic dataset of ~50,000 journal articles to explore
research output, citation patterns, and collaboration across different academic fields —
this time using **Polars** instead of Pandas.

**Topics covered:**
1. Data Loading
2. Inspection & summary statistics
3. Filtering & selection
4. Handling missing data
5. Data cleaning & standardization
6. GroupBy & aggregation
7. Joining datasets
8. Pivot tables
9. Time-series resampling
10. Visualization

## 1. Setup & Data Loading

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Polars infers types automatically; pub_date stays a string for now.
# Unlike pandas, pl.read_csv does NOT treat the literal text "nan" / "NaN" as null
# by default — we pass null_values so missing funding entries become proper nulls.
articles = pl.read_csv(
    "pandas_dataset.csv",
    null_values=["", "nan", "NaN", "NA"],
)

In [ ]:
articles.head(10)

## 2. Data Inspection & Summary Statistics

Let's examine dataset shape, types, and distributions.

In [ ]:
articles.shape

In [ ]:
# Polars equivalent of pandas .info() — show column names, types, and a preview.
articles.glimpse()

In [ ]:
# Polars .describe() covers numeric AND string columns in one table
# (unlike pandas, which needs include="object" for strings).
articles.describe()

In [ ]:
# Check for missing values
articles.null_count()

## 3. Filtering & Selection

Polars uses explicit expressions via `pl.col(...)` inside `.filter()`.
This enables lazy evaluation and query optimization.

In [ ]:
# Open-access CS papers published after 2018 with > 50 citations
# First, convert pub_date from string to Date for proper comparisons
articles = articles.with_columns(
    pl.col("pub_date").str.to_date()
)

cs_labels = ["Computer Science", "CS", "Comp Sci"]

high_impact_cs = articles.filter(
    pl.col("field").is_in(cs_labels) &
    (pl.col("pub_date") >= pl.date(2019, 1, 1)) &
    (pl.col("citations") > 50) &
    (pl.col("open_access") == True)
)

print(f"Found {len(high_impact_cs)} articles matching the criteria")
high_impact_cs.select(["article_id", "field", "pub_date", "citations", "author_count"]).head(10)

In [ ]:
# Polars also supports SQL via .sql() — a readable alternative to .filter()
result = articles.sql("""
    SELECT * FROM self
    WHERE field IN ('Computer Science', 'CS', 'Comp Sci')
      AND pub_date >= '2019-01-01'
      AND citations > 50
      AND open_access = true
""")
print(f"Same result: {len(result)} articles")

## 4. Handling Missing Data

Data in real datasets may have gaps.

Here the `funding_source` column has null values,
representing articles whose funding information wasn't recorded.

In [ ]:
# How many articles are missing funding info?
missing_funding = articles["funding_source"].null_count()
pct = 100 * missing_funding / len(articles)
print(f"Missing funding_source: {missing_funding} ({pct:.1f}%)")

In [ ]:
# Option A: Fill missing values with a label
articles = articles.with_columns(
    pl.col("funding_source").fill_null("Unknown").alias("funding_filled")
)
articles["funding_filled"].value_counts().sort("count", descending=True)

In [ ]:
# Option B: Drop rows with missing funding (careful — you lose data)
articles_complete = articles.drop_nulls(subset=["funding_source"])
print(f"Rows before: {len(articles)}, after drop_nulls: {len(articles_complete)}")

## 5. Data Cleaning & Standardization

Notice that the `field` column has inconsistent names: "Computer Science", "CS", and "Comp Sci"
all mean the same thing. Let's fix it.

In [ ]:
# See the messy values
print("Before cleaning:")
print(articles["field"].value_counts().sort("count", descending=True))

In [ ]:
# Build a mapping from messy names to canonical names
field_map = {
    "CS":       "Computer Science",
    "Comp Sci": "Computer Science",
    "Bio":      "Biology",
    "Econ":     "Economics",
    "Psych":    "Psychology",
    "Chem":     "Chemistry",
}

articles = articles.with_columns(
    pl.col("field").replace(field_map).alias("field_clean")
)

print("\nAfter cleaning:")
print(articles["field_clean"].value_counts().sort("count", descending=True))

## 6. GroupBy & Aggregation

GroupBy is arguably the most important Polars operation for analysis. It splits your data
into groups, applies an expression, and combines the results. Polars `.group_by().agg()`
is more explicit than pandas: every aggregation is spelled out as an expression.

In [ ]:
# Mean citations by field
articles.group_by("field_clean").agg(
    pl.col("citations").mean()
).sort("citations", descending=True)

In [ ]:
# Multiple aggregations at once
summary = (
    articles.group_by("field_clean")
    .agg(
        pl.len().alias("num_articles"),
        pl.col("citations").mean().alias("mean_citations"),
        pl.col("citations").median().alias("median_citations"),
        pl.col("author_count").mean().alias("mean_authors"),
        pl.col("open_access").mean().alias("pct_open_access"),
    )
    .with_columns(
        pl.col("mean_citations").round(2),
        pl.col("mean_authors").round(2),
        (pl.col("pct_open_access") * 100).round(1).alias("pct_open_access"),
    )
    .sort("mean_citations", descending=True)
)
summary

In [ ]:
# Yearly publication counts by field — long form first, then pivot wide
yearly_long = (
    articles
    .with_columns(pl.col("pub_date").dt.year().alias("year"))
    .group_by(["year", "field_clean"])
    .agg(pl.len().alias("count"))
)

yearly = yearly_long.pivot(
    on="field_clean",
    index="year",
    values="count",
).sort("year").fill_null(0)

yearly

## 7. Joining Datasets

Often your data lives in multiple tables. Polars uses `.join()` (pandas calls it `.merge()`).
Let's create a journal-level dataset and join it with our articles to see whether journal
prestige correlates with citation count.

In [ ]:
# Create a synthetic journal impact factor table
journals = pl.DataFrame({
    "field_clean": ["Computer Science", "Biology", "Physics",
                    "Economics", "Psychology", "Chemistry"],
    "avg_impact_factor": [5.2, 7.8, 6.1, 3.4, 4.5, 4.9],
    "avg_review_months": [4.2, 6.1, 5.0, 8.3, 7.5, 5.8],
})
journals

In [ ]:
# Join on the cleaned field column
merged = articles.join(journals, on="field_clean", how="left")
print(f"Shape after join: {merged.shape}")
merged.select(["article_id", "field_clean", "citations", "avg_impact_factor"]).head()

In [ ]:
# Correlation between impact factor and citations
corr = merged.select(["citations", "avg_impact_factor", "author_count"]).corr()
print("Correlation matrix:")
corr.with_columns(pl.all().round(3))

## 8. Pivot Tables

Pivot tables are a powerful way to reshape data for comparison. Let's look at how open-access
status relates to citation counts across fields.

In [ ]:
# Cast boolean to a readable label before pivoting
oa_labelled = articles.with_columns(
    pl.when(pl.col("open_access"))
      .then(pl.lit("Open Access"))
      .otherwise(pl.lit("Closed Access"))
      .alias("oa_label")
)

pivot = oa_labelled.pivot(
    on="oa_label",
    index="field_clean",
    values="citations",
    aggregate_function="mean",
).with_columns(
    pl.col("Open Access").round(1),
    pl.col("Closed Access").round(1),
).with_columns(
    ((pl.col("Open Access") / pl.col("Closed Access") - 1) * 100)
        .round(1).alias("OA Advantage (%)")
).sort("OA Advantage (%)", descending=True)

pivot

## 9. Time-Series Resampling

Since we have publication dates, we can look at trends over time. Polars uses
`.group_by_dynamic()` for time-based grouping (the equivalent of pandas `.resample()`)
and `.rolling_mean()` for smoothing.

In [ ]:
# group_by_dynamic requires a sorted temporal key
ts = articles.sort("pub_date")

# Quarterly publication counts
quarterly = (
    ts.group_by_dynamic("pub_date", every="1q")
      .agg(pl.len().alias("num_articles"))
      .with_columns(
          pl.col("num_articles").rolling_mean(window_size=4).alias("rolling_avg")
      )
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(quarterly["pub_date"], quarterly["num_articles"], alpha=0.4, label="Quarterly count")
ax.plot(quarterly["pub_date"], quarterly["rolling_avg"], linewidth=2, label="4-quarter rolling avg")
ax.set_title("Publication Volume Over Time")
ax.set_ylabel("Number of Articles")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Monthly median citations — are recent papers cited less (recency bias)?
monthly = (
    ts.group_by_dynamic("pub_date", every="1mo")
      .agg(pl.col("citations").median().alias("median_citations"))
      .with_columns(
          pl.col("median_citations").rolling_mean(window_size=6).alias("rolling_median")
      )
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly["pub_date"], monthly["median_citations"], alpha=0.5, label="Monthly")
ax.plot(monthly["pub_date"], monthly["rolling_median"], linewidth=2, color="red", label="6-month rolling median")
ax.set_title("Median Citations Over Time")
ax.set_ylabel("Median Citations")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Visualization

Polars doesn't have a built-in `.plot()` accessor like Pandas does, so we pass its
Series to Matplotlib directly (Polars Series are NumPy-compatible via `.to_numpy()`
and iterate cleanly). For interactive plots you can also use `hvplot` or `plotly`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart: articles per field
field_counts = articles["field_clean"].value_counts().sort("count")
axes[0].barh(field_counts["field_clean"], field_counts["count"], color="steelblue")
axes[0].set_title("Number of Articles by Field")
axes[0].set_xlabel("Count")

# Box plot: citation distributions by field
fields = sorted(articles["field_clean"].unique().to_list())
box_data = [
    articles.filter(pl.col("field_clean") == f)["citations"].to_numpy()
    for f in fields
]
axes[1].boxplot(box_data, labels=fields, showfliers=False)
axes[1].set_title("Citation Distribution by Field")
axes[1].set_ylabel("Citations")
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: author count vs. citations (sampled for readability)
sample = articles.sample(n=1000, seed=42)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["green" if oa else "gray" for oa in sample["open_access"]]
ax.scatter(sample["author_count"], sample["citations"], c=colors, alpha=0.4, s=20)
ax.set_xlabel("Number of Authors")
ax.set_ylabel("Citations")
ax.set_title("Author Count vs. Citations")
ax.set_yscale("log")

# Manual legend
import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color="green", label="Open Access"),
    mpatches.Patch(color="gray",  label="Closed Access"),
])
plt.tight_layout()
plt.show()

## Summary & Next Steps

In this notebook we covered the same analytical workflow as the Pandas notebook, but
using Polars. Here's a quick translation reference:

| Pandas | Polars |
|---|---|
| `pd.read_csv()` | `pl.read_csv()` |
| `df.info()` | `df.glimpse()` |
| `df.isnull().sum()` | `df.null_count()` |
| `df[cond]`, `df.query()` | `df.filter(pl.col(...))`, `df.sql()` |
| `df.fillna(x)` | `df.with_columns(pl.col(c).fill_null(x))` |
| `df.dropna()` | `df.drop_nulls()` |
| `s.replace(mapping)` | `pl.col(c).replace(mapping)` |
| `df.groupby().agg()` | `df.group_by().agg()` |
| `df.merge()` | `df.join()` |
| `df.pivot_table()` | `df.pivot()` |
| `df.resample()` | `df.group_by_dynamic()` |
| `s.rolling(n).mean()` | `pl.col(c).rolling_mean(window_size=n)` |

**Why Polars?**
- Multi-threaded execution by default → much faster on large data.
- Lazy API (`pl.scan_csv(...).collect()`) enables query optimization.
- Explicit expressions make code easier to reason about and compose.
- Memory-efficient: typically uses 2–5× less RAM than Pandas.

**Next steps:**
- Rewrite the above using the lazy API (`pl.scan_csv` + `.collect()`) and compare performance.
- For datasets that don't fit in memory, try **[Polars streaming](https://docs.pola.rs/user-guide/concepts/streaming/)** or **Dask**.